# Simsalabim — End-to-End Training on Google Colab

Full pipeline from raw WAVs to a trained song-identification encoder, runnable top-to-bottom on a Colab GPU. Every step prints what it produced so you can verify before moving on.

**Sections:**

| # | What it does |
|---|---|
| 1 | Runtime check (GPU) |
| 2 | Mount Google Drive |
| 3 | Unpack the dataset zip from Drive → local SSD |
| 4 | Clone the repo + install Python deps |
| 5 | Import `src/` and load YAML configs |
| 6 | Reset stale artifacts (optional) |
| 7 | Inspect the manifest |
| 8 | Make song-level splits (train / val / test) |
| 9 | Materialize per-split manifests + ID maps |
| 10 | Compute per-bin mel mean/std (one-shot) |
| 11 | Build `TakesDataset`s (train + eval) |
| 12 | Visualize a single sample (waveform + mel + f0) |
| 13 | Build encoder + Sub-center ArcFace loss + optimizer |
| 14 | Smoke-train one epoch |
| 15 | Full training loop with periodic retrieval eval |
| 16 | Training curves |
| 17 | Load best checkpoint + build FAISS gallery |
| 18 | Final per-style retrieval metrics |
| 19 | Inference on a single WAV |
| 20 | Save everything back to Drive |

**Required Drive layout:**

```
My Drive/SimlabimAI/dataset_data.zip       ← created by `zip -r dataset_data.zip data/` in dataset/
My Drive/SimlabimAI/model_artifacts/       ← created automatically; persists checkpoints across sessions
```

All the *why*s for the design choices live in `model/PLAN.md` in the repo — read it before tweaking hyperparameters.

## 1 — Runtime check

First confirm we're on a GPU. `Runtime → Change runtime type → GPU` if not.

In [ ]:
import subprocess, sys
print('python:', sys.version.split()[0])
try:
    print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader']).decode().strip())
except Exception as err:
    print('!! no GPU detected — switch to a GPU runtime:', err)

## 2 — Mount Google Drive

Click the auth link, pick your account, paste the token back. After this, `My Drive` is at `/content/drive/MyDrive/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/SimlabimAI')
DATASET_ZIP = DRIVE_ROOT / 'dataset_data.zip'        # made locally with `zip -r dataset_data.zip data`
ARTIFACTS_ROOT = DRIVE_ROOT / 'model_artifacts'       # persistent outputs (checkpoints, stats, gallery, ...)
ARTIFACTS_ROOT.mkdir(parents=True, exist_ok=True)

assert DATASET_ZIP.exists(), f'missing {DATASET_ZIP} — upload your dataset_data.zip to Drive first'
print(f'zip OK: {DATASET_ZIP}  ({DATASET_ZIP.stat().st_size/1e6:.1f} MB)')
print(f'artifacts dir: {ARTIFACTS_ROOT}')

## 3 — Unpack the dataset to local SSD

Reading directly from Drive is slow and flaky (every WAV open is a network round-trip). We extract once into `/content/dataset_data/` on the local SSD and read from there for the rest of the notebook. ~10 s for 50 wavs.

In [ ]:
import shutil, zipfile, time

LOCAL_DATASET = Path('/content/dataset_data')

if LOCAL_DATASET.exists():
    print(f'reusing existing {LOCAL_DATASET}')
else:
    t0 = time.time()
    extract_into = Path('/content/_extract_tmp')
    if extract_into.exists():
        shutil.rmtree(extract_into)
    extract_into.mkdir()
    with zipfile.ZipFile(DATASET_ZIP) as z:
        z.extractall(extract_into)
    # the zip is `data/...` at top level
    src = extract_into / 'data'
    assert src.exists(), f'zip layout unexpected — got {list(extract_into.iterdir())}'
    src.rename(LOCAL_DATASET)
    shutil.rmtree(extract_into, ignore_errors=True)
    print(f'extracted in {time.time()-t0:.1f}s → {LOCAL_DATASET}')

DATASET_ROOT = LOCAL_DATASET
MANIFEST_CSV = DATASET_ROOT / 'manifest.csv'

n_wav = sum(1 for _ in LOCAL_DATASET.rglob('*.wav'))
assert MANIFEST_CSV.exists(), f'missing {MANIFEST_CSV} — bad zip?'
print(f'manifest: {MANIFEST_CSV}\n{n_wav} wav files under {LOCAL_DATASET}/raw_audio/')

## 4 — Clone the repo and install dependencies

The repo gives us `model/src/` (all the Python modules) and `shared/` (the canonical JSON invariants the modules validate against).

If you change `model/src/` on your laptop, push to GitHub and re-run this cell — `git pull --ff-only` brings the changes in without a runtime restart (unless you change class signatures).

In [ ]:
REPO_URL = 'https://github.com/Gustavoo-Pacheco/SimlabimAI.git'
REPO_DIR = Path('/content/SimlabimAI')

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull --ff-only

MODEL_DIR = REPO_DIR / 'model'
assert MODEL_DIR.exists() and (REPO_DIR / 'shared').exists(), 'repo layout mismatch'
print('repo OK at', REPO_DIR)

In [ ]:
# Colab ships torch + numpy + matplotlib. Only pip the missing pieces (~90 s on a cold runtime).
!pip install -q audiomentations pesto-pitch pyloudnorm pytorch-metric-learning faiss-cpu fast_mp3_augment soundfile pyyaml tqdm

## 5 — Import the `src/` package and load configs

All hyperparameters live in `model/configs/*.yaml`. The notebook reads them once and passes them down — never hardcode.

In [ ]:
import sys
if str(MODEL_DIR) not in sys.path:
    sys.path.insert(0, str(MODEL_DIR))

import yaml, json
from src.io import load_wav, EXPECTED_SR
from src.preproc import SAMPLE_RATE, crop_or_pad, normalize_loudness, trim_by_confidence
from src.representation import (
    MelConfig, MelExtractor, PestoConfig, PestoExtractor,
    coarse_confidence_for_trim, normalize_log_mel,
)
from src.augment_waveform import build_waveform_augmenter
from src.augment_mel import SpecAugmenter, SpecAugmentConfig
from src.dataset import load_manifest, build_song_id_map, TakesDataset
from src.splits import make_splits, save_splits, load_splits, is_gallery_take
from src.encoder import TwoStreamEncoder, EncoderConfig
from src.loss import SongArcFaceLoss, ArcFaceConfig
from src.train import (
    make_loader, train_one_epoch, evaluate_retrieval,
    save_checkpoint, load_checkpoint, TrainState,
)
from src.enroll import build_gallery, Gallery
from src.infer import infer_wav

PREPROC   = yaml.safe_load((MODEL_DIR / 'configs/preproc.yaml').read_text())
MODEL_CFG = yaml.safe_load((MODEL_DIR / 'configs/model.yaml').read_text())
TRAIN_CFG = yaml.safe_load((MODEL_DIR / 'configs/train.yaml').read_text())
INFER_CFG = yaml.safe_load((MODEL_DIR / 'configs/infer.yaml').read_text())

print(f'imports OK — sample rate = {EXPECTED_SR} Hz')
print(f'configs loaded: preproc/model/train/infer')

## 6 — Reset stale artifacts (optional)

If you ran the notebook against a partial/broken dataset earlier, `splits.json`, `stats.json`, and old `checkpoints/` on Drive are stale. Set `RESET = True` to wipe them and start clean.

In [ ]:
RESET = False    # set True ONCE when you want a fresh run, then flip back to False

if RESET:
    for p in sorted(ARTIFACTS_ROOT.rglob('*'), reverse=True):
        if p.is_file(): p.unlink()
        elif p.is_dir(): p.rmdir()
    print(f'wiped {ARTIFACTS_ROOT}')
else:
    existing = sorted(p.relative_to(ARTIFACTS_ROOT) for p in ARTIFACTS_ROOT.rglob('*') if p.is_file())
    print(f'{len(existing)} existing artifact files (set RESET=True to wipe):')
    for p in existing[:10]:
        print(' -', p)

## 7 — Inspect the manifest

`manifest.csv` is one row per take. We keep `approved` + `pending` for now (the toy dataset has nothing approved yet); switch to `('approved',)` once moderation catches up.

In [ ]:
import csv
from collections import Counter

ACCEPTED_STATUSES = ('approved', 'pending')

with MANIFEST_CSV.open() as f:
    rows = [r for r in csv.DictReader(f) if r['status'] in ACCEPTED_STATUSES]

songs = Counter(r['song_slug'] for r in rows)
styles = Counter(r['style'] for r in rows)
print(f'takes accepted: {len(rows)}')
print(f'unique songs:   {len(songs)}')
print(f'style mix:      {dict(styles)}')
print(f'songs with >1 take: {sum(1 for v in songs.values() if v > 1)}/{len(songs)}')
print(f'top by take count: {songs.most_common(5)}')

## 8 — Song-level train / val / test splits

**Unit of split = song**, never take. We have to evaluate generalization to *unseen songs* — taking different takes of the same song into train and val leaks identity. Splits are persisted to Drive so they stay stable across sessions.

In [ ]:
SPLITS_PATH = ARTIFACTS_ROOT / 'splits.json'
SPLIT_SEED = 0
RATIOS = (0.7, 0.15, 0.15)

if SPLITS_PATH.exists():
    splits = load_splits(SPLITS_PATH)
    print(f'loaded existing splits from {SPLITS_PATH}')
else:
    splits = make_splits([r['song_slug'] for r in rows], ratios=RATIOS, seed=SPLIT_SEED)
    save_splits(splits, SPLITS_PATH)
    print(f'wrote new splits to {SPLITS_PATH}')

print(f'train: {len(splits.train)} songs')
print(f'val:   {len(splits.val)} songs — {splits.val}')
print(f'test:  {len(splits.test)} songs — {splits.test}')

## 9 — Per-split manifests + ID maps

The encoder learns song IDs from the train set only (these are the ArcFace classes). Val/test use separate ID spaces and are evaluated by retrieval, not classification.

In [ ]:
all_manifest = load_manifest(MANIFEST_CSV, DATASET_ROOT, statuses=ACCEPTED_STATUSES)

train_rows = [r for r in all_manifest if r.song_slug in set(splits.train)]
val_rows   = [r for r in all_manifest if r.song_slug in set(splits.val)]
test_rows  = [r for r in all_manifest if r.song_slug in set(splits.test)]

song_id_map = build_song_id_map(train_rows)
val_id_map  = build_song_id_map(val_rows)
test_id_map = build_song_id_map(test_rows)

print(f'train: {len(train_rows)} takes across {len(song_id_map)} songs   (= ArcFace num_classes)')
print(f'val:   {len(val_rows)} takes across {len(val_id_map)} songs')
print(f'test:  {len(test_rows)} takes across {len(test_id_map)} songs')

## 10 — Mel statistics (per-bin mean/std, one-shot)

Each log-mel bin gets `(x - mean) / std` applied so the CNN sees zero-mean unit-std inputs. Stats are computed over train only, **without augmentation**, with a center crop. PESTO + LUFS per take → ~2–4 s/take on T4. Saved to Drive; only re-run when splits or preproc change.

In [ ]:
import torch
from tqdm.auto import tqdm

STATS_PATH = ARTIFACTS_ROOT / 'stats.json'

def compute_mel_stats(rows, preproc_cfg):
    mel_cfg = MelConfig(**{k: preproc_cfg['mel'][k] for k in (
        'n_fft', 'win_length', 'hop_length', 'n_mels', 'f_min', 'f_max', 'mel_scale', 'power'
    )})
    mel_extractor = MelExtractor(mel_cfg)
    duration_samples = int(preproc_cfg['crop']['duration_s'] * SAMPLE_RATE)
    trim = preproc_cfg['trim']

    n_mels = mel_cfg.n_mels
    sum_x  = torch.zeros(n_mels, dtype=torch.float64)
    sum_x2 = torch.zeros(n_mels, dtype=torch.float64)
    count = 0

    for r in tqdm(rows, desc='mel stats'):
        wav = load_wav(r.audio_path)
        conf = coarse_confidence_for_trim(wav, step_size_ms=trim['step_size_ms'])
        wav = trim_by_confidence(wav, conf,
            frame_hop_ms=trim['step_size_ms'],
            threshold=trim['confidence_threshold'],
            margin_ms=trim['margin_ms'])
        wav = normalize_loudness(wav, target_lufs=preproc_cfg['loudness']['target_lufs'])
        wav = crop_or_pad(wav, duration_samples, mode='center')
        mel = mel_extractor(wav).to(torch.float64)
        sum_x  += mel.sum(dim=1)
        sum_x2 += (mel * mel).sum(dim=1)
        count += mel.shape[1]

    mean = (sum_x / count)
    var = (sum_x2 / count) - mean ** 2
    std = torch.sqrt(var.clamp_min(0.0))
    return {'n_mels': n_mels, 'mean': mean.tolist(), 'std': std.tolist(),
            'frames_aggregated': count, 'takes': len(rows)}

if STATS_PATH.exists():
    print(f'loaded existing stats from {STATS_PATH}')
else:
    s = compute_mel_stats(train_rows, PREPROC)
    STATS_PATH.write_text(json.dumps(s, indent=2))
    print(f'wrote {STATS_PATH}')

stats = json.loads(STATS_PATH.read_text())
print(f'n_mels={stats["n_mels"]}  takes={stats["takes"]}  frames={stats["frames_aggregated"]}')

## 11 — Build the datasets

Train gets waveform-level augmentations (pitch shift, time stretch, noise, EQ, …) + SpecAugment. Val/test get neither — they must see the deterministic pipeline so evaluation is reproducible.

You'll see warnings about `background_noise` / `ir_reverb` being skipped — that's expected unless you've uploaded an IR / noise folder and pointed `configs/preproc.yaml` at it.

In [ ]:
mel_cfg = MelConfig(**{k: PREPROC['mel'][k] for k in (
    'n_fft', 'win_length', 'hop_length', 'n_mels', 'f_min', 'f_max', 'mel_scale', 'power'
)})
pesto_cfg = PestoConfig(
    step_size_ms=PREPROC['pesto']['step_size_ms'],
    confidence_threshold=PREPROC['pesto']['confidence_threshold'],
    median_subtract=PREPROC['pesto']['median_subtract'],
)

waveform_aug = build_waveform_augmenter(PREPROC['augment']['waveform'])
sa = PREPROC['augment']['mel']['spec_augment']
spec_aug = SpecAugmenter(SpecAugmentConfig(
    time_mask_p=sa['time_mask']['p'], time_mask_count=sa['time_mask']['count'], time_mask_max=sa['time_mask']['max_width'],
    freq_mask_p=sa['freq_mask']['p'], freq_mask_count=sa['freq_mask']['count'], freq_mask_max=sa['freq_mask']['max_width'],
))

common = dict(
    mel_cfg=mel_cfg, pesto_cfg=pesto_cfg,
    mel_stats_path=STATS_PATH,
    crop_duration_s=PREPROC['crop']['duration_s'],
    trim_step_size_ms=PREPROC['trim']['step_size_ms'],
    trim_confidence_threshold=PREPROC['trim']['confidence_threshold'],
    trim_margin_ms=PREPROC['trim']['margin_ms'],
    target_lufs=PREPROC['loudness']['target_lufs'],
)

train_ds = TakesDataset(train_rows, song_id_map, train=True,
                        waveform_augmenter=waveform_aug, spec_augmenter=spec_aug, **common)
val_ds   = TakesDataset(val_rows,   val_id_map,   train=False, **common)
test_ds  = TakesDataset(test_rows,  test_id_map,  train=False, **common)

print(f'len(train)={len(train_ds)}  len(val)={len(val_ds)}  len(test)={len(test_ds)}')

## 12 — Visualize one sample

Three plots: raw waveform, normalized log-mel, f0 contour. If the mel is solid black or f0 is flat zero, something upstream is broken — stop and debug before training.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

sample = val_ds[0] if len(val_ds) else train_ds[0]
mel = sample['mel'].numpy()
f0  = sample['f0'].numpy()
raw_path = (val_rows[0] if val_rows else train_rows[0]).audio_path
raw = load_wav(raw_path).numpy()

fig, axes = plt.subplots(3, 1, figsize=(12, 7))
axes[0].plot(np.arange(len(raw)) / SAMPLE_RATE, raw, lw=0.4)
axes[0].set_title(f'raw waveform — {sample["take_id"]} ({sample["style"]})')
axes[0].set_xlabel('s'); axes[0].set_ylabel('amplitude')

axes[1].imshow(mel, origin='lower', aspect='auto', cmap='magma')
axes[1].set_title(f'log-mel (normalized)  shape={mel.shape}')
axes[1].set_ylabel('mel bin')

axes[2].plot(f0[0], label='pitch (semitones, median-subtracted)')
axes[2].plot(f0[1], label='confidence', alpha=0.7)
axes[2].set_title(f'f0 stream  shape={f0.shape}')
axes[2].set_xlabel('frame'); axes[2].legend()
plt.tight_layout(); plt.show()

## 13 — Encoder + Sub-center ArcFace loss + optimizer

- **Encoder:** mel → ResNet-18 (1-channel stem) → GAP [512]; f0 → 1D-CNN → GAP [128]; fusion → L2-normalized 256-d.
- **Loss:** Sub-center ArcFace, one class per train song, 3 sub-centers per class (naturally absorbs cantar/cantarolar/assobiar). Loss parameters are learnable → they go into the optimizer too.
- **Optimizer:** AdamW with cosine annealing + warmup.

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)

encoder = TwoStreamEncoder(EncoderConfig(
    embedding_dim=MODEL_CFG['encoder']['embedding_dim'],
    fusion_hidden=MODEL_CFG['encoder']['fusion_hidden'],
    dropout=MODEL_CFG['encoder']['dropout'],
    f0_in_channels=MODEL_CFG['encoder']['f0_in_channels'],
)).to(device)

loss_fn = SongArcFaceLoss(ArcFaceConfig(
    num_classes=len(song_id_map),
    embedding_size=MODEL_CFG['encoder']['embedding_dim'],
    margin=MODEL_CFG['arcface']['margin'],
    scale=MODEL_CFG['arcface']['scale'],
    sub_centers=MODEL_CFG['arcface']['sub_centers'],
)).to(device)

params = list(encoder.parameters()) + list(loss_fn.parameters())
optimizer = AdamW(params,
                  lr=TRAIN_CFG['optimizer']['lr'],
                  weight_decay=TRAIN_CFG['optimizer']['weight_decay'])

print(f'encoder params: {sum(p.numel() for p in encoder.parameters())/1e6:.2f} M')
print(f'arcface classes: {len(song_id_map)}  sub_centers: {MODEL_CFG["arcface"]["sub_centers"]}')
print(f'device: {device}')

## 14 — Smoke train (1 epoch)

Sanity check before the real loop: one epoch, watch the loss. With a tiny dataset the absolute value is meaningless — what matters is that it **moves**.

In [ ]:
BATCH = min(TRAIN_CFG['loop']['batch_size'], max(2, len(train_ds) // 2))
train_loader = make_loader(train_ds, batch_size=BATCH, shuffle=True,
                           num_workers=TRAIN_CFG['loop']['num_workers'])

smoke = train_one_epoch(
    encoder, loss_fn, optimizer, train_loader,
    device=device,
    use_amp=TRAIN_CFG['loop']['mixed_precision'],
    grad_clip=TRAIN_CFG['loop']['grad_clip'],
    mixup_alpha=TRAIN_CFG['mixup']['alpha'], mixup_p=TRAIN_CFG['mixup']['p'],
    log_every=5,
)
print('smoke epoch mean loss:', smoke['loss'])

## 15 — Full training loop

Cosine schedule + warmup, periodic retrieval eval on val, checkpoint best-by-mAP@10 to Drive. With ~50 takes a few dozen epochs is plenty; for real datasets (hundreds of songs × multiple takes), aim for 100+.

You can interrupt the cell at any time — `last.pt` is updated every epoch and `best_map.pt` every improvement, so progress is never lost.

In [ ]:
EPOCHS = 30
EVAL_EVERY = max(1, TRAIN_CFG['loop']['eval_every'])
CKPT_DIR = ARTIFACTS_ROOT / 'checkpoints'
CKPT_DIR.mkdir(exist_ok=True)

steps_per_epoch = max(1, len(train_loader))
total_steps = steps_per_epoch * EPOCHS
warmup_steps = int(total_steps * TRAIN_CFG['schedule']['warmup_ratio'])

def lr_lambda(step):
    if step < warmup_steps:
        return step / max(1, warmup_steps)
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.5 * (1.0 + np.cos(np.pi * progress))

scheduler = LambdaLR(optimizer, lr_lambda)
state = TrainState()
history = []

for epoch in range(1, EPOCHS + 1):
    print(f'\n=== epoch {epoch}/{EPOCHS} ===')
    metrics = train_one_epoch(
        encoder, loss_fn, optimizer, train_loader,
        device=device, scheduler=scheduler,
        use_amp=TRAIN_CFG['loop']['mixed_precision'],
        grad_clip=TRAIN_CFG['loop']['grad_clip'],
        mixup_alpha=TRAIN_CFG['mixup']['alpha'], mixup_p=TRAIN_CFG['mixup']['p'],
        log_every=20,
    )
    state.epoch = epoch
    record = {'epoch': epoch, 'train_loss': metrics['loss']}

    if epoch % EVAL_EVERY == 0 and len(val_ds) >= 2:
        eval_out = evaluate_retrieval(encoder, val_ds,
            is_gallery_fn=lambda tid: is_gallery_take(tid, seed=SPLIT_SEED),
            device=device, batch_size=32, k_for_map=10)
        overall = eval_out['overall']
        record.update({f'val_{k}': overall[k] for k in ('top1', 'top5', 'mrr', 'map_at_10')})
        record['val_n_query'] = eval_out['n_query']
        record['val_n_gallery'] = eval_out['n_gallery']
        print(f'  val: top1={overall["top1"]:.3f}  top5={overall["top5"]:.3f}  '
              f'mrr={overall["mrr"]:.3f}  mAP@10={overall["map_at_10"]:.3f}  '
              f'(N_q={eval_out["n_query"]}, N_g={eval_out["n_gallery"]})')
        if overall['map_at_10'] > state.best_map:
            state.best_map = overall['map_at_10']
            save_checkpoint(CKPT_DIR / 'best_map.pt',
                            model=encoder, loss_fn=loss_fn,
                            optimizer=optimizer, scheduler=scheduler, state=state)
            print('  ★ new best mAP@10 — saved checkpoint')

    history.append(record)
    save_checkpoint(CKPT_DIR / 'last.pt',
                    model=encoder, loss_fn=loss_fn,
                    optimizer=optimizer, scheduler=scheduler, state=state)

print('\ntraining done. best val mAP@10:', state.best_map)

## 16 — Training curves

In [ ]:
import pandas as pd

df = pd.DataFrame(history)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(df['epoch'], df['train_loss'], marker='o')
axes[0].set_title('train loss'); axes[0].set_xlabel('epoch'); axes[0].grid(True, alpha=0.3)
if 'val_map_at_10' in df.columns:
    eval_df = df.dropna(subset=['val_map_at_10'])
    axes[1].plot(eval_df['epoch'], eval_df['val_map_at_10'], marker='o', label='mAP@10')
    axes[1].plot(eval_df['epoch'], eval_df['val_top1'],     marker='s', label='top-1')
    axes[1].plot(eval_df['epoch'], eval_df['val_top5'],     marker='^', label='top-5')
    axes[1].set_title('val retrieval'); axes[1].set_xlabel('epoch'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 17 — Load best checkpoint + build FAISS gallery

For eval and inference we always use the **best** checkpoint, not the last. The gallery indexes one vector per val take (no per-song averaging — variance across takes is the signal).

In [ ]:
best_path = CKPT_DIR / 'best_map.pt'
if best_path.exists():
    _ = load_checkpoint(best_path, model=encoder, loss_fn=loss_fn)
    print('loaded', best_path)
else:
    print('no best checkpoint — using current weights')

id_to_slug = {v: k for k, v in val_id_map.items()}
val_gallery = build_gallery(encoder, val_ds, id_to_slug=id_to_slug, device=device, batch_size=32)
GALLERY_PATH = ARTIFACTS_ROOT / 'val_gallery.npz'
val_gallery.save(GALLERY_PATH)
print(f'gallery: {val_gallery.embeddings.shape}  →  {GALLERY_PATH}')

## 18 — Final per-style retrieval metrics

Broken down by query style — that's the diagnostic that tells us where the model is weakest (PLAN.md [12]).

In [ ]:
final = evaluate_retrieval(encoder, val_ds,
    is_gallery_fn=lambda tid: is_gallery_take(tid, seed=SPLIT_SEED),
    device=device, batch_size=32, k_for_map=10)
print('OVERALL:', final['overall'])
print()
for style, m in final['per_style'].items():
    print(f'{style:<12}  top1={m["top1"]:.3f}  top5={m["top5"]:.3f}  '
          f'mAP@10={m["map_at_10"]:.3f}  (N={m["n_queries"]})')

(ARTIFACTS_ROOT / 'eval_val.json').write_text(json.dumps(final, indent=2))

## 19 — Inference on a single WAV

Full runtime pipeline: load → trim → LUFS → sliding 10 s/5 s windows → mel + f0 → embed → FAISS top-3 per window → sum-of-top-3 aggregator → final top-K.

In [ ]:
mel_mean = torch.tensor(stats['mean'], dtype=torch.float32)
mel_std  = torch.tensor(stats['std'],  dtype=torch.float32)

demo_path = (val_rows[0] if val_rows else train_rows[0]).audio_path
result = infer_wav(
    demo_path,
    model=encoder, gallery=val_gallery,
    mel_cfg=mel_cfg, pesto_cfg=pesto_cfg,
    mel_mean=mel_mean, mel_std=mel_std,
    window_s=INFER_CFG['window_s'], hop_s=INFER_CFG['hop_s'],
    top_k_per_window=INFER_CFG['top_k_per_window'],
    top_n_return=INFER_CFG['top_n_return'],
    device=device,
    target_lufs=PREPROC['loudness']['target_lufs'],
    trim_step_size_ms=PREPROC['trim']['step_size_ms'],
    trim_confidence_threshold=PREPROC['trim']['confidence_threshold'],
    trim_margin_ms=PREPROC['trim']['margin_ms'],
)
print(f'WAV: {demo_path.name}  windows={result.n_windows}')
for rank, (slug, score) in enumerate(result.top, 1):
    print(f'  {rank}. {slug:<32} score={score:.3f}')

## 20 — Save everything to Drive

By now Drive already has `splits.json`, `stats.json`, `checkpoints/best_map.pt`, `checkpoints/last.pt`, `val_gallery.npz`, `eval_val.json`. We also dump resolved configs + history for reproducibility, then list what's there.

In [ ]:
(ARTIFACTS_ROOT / 'configs_resolved.json').write_text(json.dumps({
    'preproc': PREPROC, 'model': MODEL_CFG, 'train': TRAIN_CFG, 'infer': INFER_CFG,
    'split_seed': SPLIT_SEED, 'ratios': list(RATIOS), 'epochs_run': len(history),
    'song_id_map_size': len(song_id_map),
}, indent=2))
(ARTIFACTS_ROOT / 'history.json').write_text(json.dumps(history, indent=2))

import os
print(f'artifacts in {ARTIFACTS_ROOT}:')
for p in sorted(ARTIFACTS_ROOT.rglob('*')):
    if p.is_file():
        print(f'  {p.relative_to(ARTIFACTS_ROOT)}  ({os.path.getsize(p)/1024:.1f} KB)')

---

## Next steps

- **Grow the dataset.** ArcFace needs `N_train ≫ N_catalogue` (PLAN.md [premissa de dataset]). With ~50 takes, all metrics are noisy and overfit.
- **Calibrate the rejection threshold τ** on val once you have enough out-of-catalogue negatives (PLAN.md [11.3]).
- **Run the test set** with frozen splits/encoder for the final reportable numbers — same flow as §17–§18 but against `test_ds`.
- **Cross-style breakdown** (e.g. query in `cantar`, gallery only in `assobiar`) — `evaluate_retrieval` already breaks down by query style; restricting the gallery is a 5-line extension.
- **Add background-noise + IR augmentations.** Upload a folder of noise WAVs and one of IR WAVs to Drive, then edit `configs/preproc.yaml` (`augment.waveform.background_noise.path`, `augment.waveform.ir_reverb.path`). Push and rerun §4.
- **Quick resume after disconnect:** §1 → §2 → §3 → §4 → §5 → skip to §11 → §13 → then `load_checkpoint(CKPT_DIR/'last.pt', model=encoder, loss_fn=loss_fn, optimizer=optimizer)` → resume §15 from where you left off.

## 21 — Record from your browser and identify the song

Runs the **same** sliding-window inference as §19 (cosine similarity against the FAISS gallery), but on a clip you record live. The audio is captured via the browser, converted to 16 kHz mono PCM-16 WAV with ffmpeg (so it matches `shared/wav.json`), and fed into `infer_wav`.

Click **Start**, sing / hum / whistle for at least 10 s, click **Stop**, then wait for the result.

Requires §17 (FAISS gallery) to have run.

In [ ]:
import base64, subprocess, uuid
from pathlib import Path
from IPython.display import Javascript, display
from google.colab import output as colab_output

REC_DIR = Path('/content/recordings'); REC_DIR.mkdir(exist_ok=True)

RECORDER_JS = """
async function recordAudio() {
  const btnStart = document.createElement('button');
  const btnStop  = document.createElement('button');
  const status   = document.createElement('span');
  btnStart.textContent = '\u25CF Start recording';
  btnStop.textContent  = '\u25A0 Stop';
  btnStop.disabled = true;
  Object.assign(btnStart.style, {marginRight: '8px', padding: '6px 12px'});
  Object.assign(btnStop.style,  {marginRight: '8px', padding: '6px 12px'});
  status.style.marginLeft = '8px';
  document.body.appendChild(btnStart);
  document.body.appendChild(btnStop);
  document.body.appendChild(status);

  const stream = await navigator.mediaDevices.getUserMedia({audio: true});
  const recorder = new MediaRecorder(stream);
  const chunks = [];
  recorder.ondataavailable = (e) => chunks.push(e.data);
  const startedAt = () => new Date();
  let t0 = null;
  const tick = () => { if (t0) status.textContent = ' recording ' + ((new Date()-t0)/1000).toFixed(1) + 's'; };
  const interval = setInterval(tick, 100);

  await new Promise((resolve) => {
    btnStart.onclick = () => { recorder.start(); t0 = startedAt(); btnStart.disabled = true; btnStop.disabled = false; status.textContent = ' recording...'; };
    btnStop.onclick  = () => { recorder.stop(); btnStop.disabled = true; resolve(); };
  });
  await new Promise((r) => recorder.onstop = r);
  clearInterval(interval);
  stream.getTracks().forEach(t => t.stop());

  const blob = new Blob(chunks);
  status.textContent = ' encoding ' + (blob.size/1024).toFixed(1) + ' KB...';
  const buf = await blob.arrayBuffer();
  const bytes = new Uint8Array(buf);
  let binary = ''; for (let i = 0; i < bytes.length; i++) binary += String.fromCharCode(bytes[i]);
  status.textContent = ' done — running inference...';
  return btoa(binary);
}
recordAudio();
"""

display(Javascript(RECORDER_JS))
b64 = colab_output.eval_js('recordAudio()', timeout_sec=600)
raw_bytes = base64.b64decode(b64)

uid = uuid.uuid4().hex[:8]
raw_path = REC_DIR / f'rec_{uid}.webm'
wav_path = REC_DIR / f'rec_{uid}.wav'
raw_path.write_bytes(raw_bytes)
print(f'saved raw recording: {raw_path}  ({len(raw_bytes)/1024:.1f} KB)')

# Convert to the shared/wav.json invariant: 16 kHz mono PCM-16 WAV.
subprocess.run([
    'ffmpeg', '-y', '-loglevel', 'error',
    '-i', str(raw_path),
    '-ac', '1', '-ar', '16000', '-sample_fmt', 's16',
    str(wav_path),
], check=True)
print(f'converted to 16 kHz mono PCM-16: {wav_path}  ({wav_path.stat().st_size/1024:.1f} KB)')

result = infer_wav(
    wav_path,
    model=encoder, gallery=val_gallery,
    mel_cfg=mel_cfg, pesto_cfg=pesto_cfg,
    mel_mean=mel_mean, mel_std=mel_std,
    window_s=INFER_CFG['window_s'], hop_s=INFER_CFG['hop_s'],
    top_k_per_window=INFER_CFG['top_k_per_window'],
    top_n_return=INFER_CFG['top_n_return'],
    device=device,
    target_lufs=PREPROC['loudness']['target_lufs'],
    trim_step_size_ms=PREPROC['trim']['step_size_ms'],
    trim_confidence_threshold=PREPROC['trim']['confidence_threshold'],
    trim_margin_ms=PREPROC['trim']['margin_ms'],
)
print(f'\n=== top {len(result.top)} matches (windows={result.n_windows}) ===')
for rank, (slug, score) in enumerate(result.top, 1):
    bar = '\u2588' * int(min(40, max(0, score) * 8))
    print(f'  {rank}. {slug:<32} {score:6.3f}  {bar}')